<a href="https://colab.research.google.com/github/magish-gautam/Magish-Concept-of-AI/blob/Workshop-2/Worksheet8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_wine, load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error


In [24]:
class CustomDecisionTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        num_samples, num_features = X.shape
        unique_classes = np.unique(y)

        if len(unique_classes) == 1:
            return {"class": unique_classes[0]}

        if self.max_depth and depth >= self.max_depth:
            return {"class": np.bincount(y).argmax()}

        best_gain = -1
        best_split = None

        for feature in range(num_features):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left = y[X[:, feature] <= threshold]
                right = y[X[:, feature] > threshold]
                if len(left) == 0 or len(right) == 0:
                    continue

                gain = self._information_gain(y, left, right)
                if gain > best_gain:
                    best_gain = gain
                    best_split = (feature, threshold)

        if best_split is None:
            return {"class": np.bincount(y).argmax()}

        feature, threshold = best_split
        left_idx = X[:, feature] <= threshold
        right_idx = X[:, feature] > threshold

        return {
            "feature": feature,
            "threshold": threshold,
            "left": self._build_tree(X[left_idx], y[left_idx], depth+1),
            "right": self._build_tree(X[right_idx], y[right_idx], depth+1)
        }

    def _information_gain(self, parent, left, right):
        return self._entropy(parent) - (
            len(left)/len(parent)*self._entropy(left)
            + len(right)/len(parent)*self._entropy(right)
        )

    def _entropy(self, y):
        probs = np.bincount(y) / len(y)
        return -np.sum(probs * np.log2(probs + 1e-9))

    def predict(self, X):
        return [self._predict_row(row, self.tree) for row in X]

    def _predict_row(self, row, node):
        if "class" in node:
            return node["class"]
        if row[node["feature"]] <= node["threshold"]:
            return self._predict_row(row, node["left"])
        else:
            return self._predict_row(row, node["right"])


In [25]:
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [26]:
custom_tree = CustomDecisionTree(max_depth=3)
custom_tree.fit(X_train, y_train)

y_pred_custom = custom_tree.predict(X_test)
custom_accuracy = accuracy_score(y_test, y_pred_custom)

print("Custom Decision Tree Accuracy:", custom_accuracy)


Custom Decision Tree Accuracy: 1.0


In [27]:
sk_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
sk_tree.fit(X_train, y_train)

y_pred_sk = sk_tree.predict(X_test)
sk_accuracy = accuracy_score(y_test, y_pred_sk)

print("Scikit-learn Decision Tree Accuracy:", sk_accuracy)


Scikit-learn Decision Tree Accuracy: 1.0


In [28]:
print("Accuracy Comparison")
print("Custom Tree:", custom_accuracy)
print("Scikit-learn Tree:", sk_accuracy)


Accuracy Comparison
Custom Tree: 1.0
Scikit-learn Tree: 1.0


In [29]:
wine = load_wine()
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [30]:
dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

dt_pred = dt.predict(X_test)
rf_pred = rf.predict(X_test)

print("Decision Tree F1:", f1_score(y_test, dt_pred, average='weighted'))
print("Random Forest F1:", f1_score(y_test, rf_pred, average='weighted'))


Decision Tree F1: 0.9439974457215836
Random Forest F1: 1.0


In [22]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring="f1_weighted",
    cv=5
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)


Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}


In [31]:
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [32]:
dt_reg = DecisionTreeRegressor(random_state=42)
rf_reg = RandomForestRegressor(random_state=42)

dt_reg.fit(X_train, y_train)
rf_reg.fit(X_train, y_train)

dt_pred = dt_reg.predict(X_test)
rf_pred = rf_reg.predict(X_test)

print("Decision Tree MSE:", mean_squared_error(y_test, dt_pred))
print("Random Forest MSE:", mean_squared_error(y_test, rf_pred))


Decision Tree MSE: 4976.797752808989
Random Forest MSE: 2952.0105887640448


In [33]:
param_dist = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10]
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_dist,
    n_iter=10,
    scoring="neg_mean_squared_error",
    cv=5,
    random_state=42
)

random_search.fit(X_train, y_train)
print("Best Regression Parameters:", random_search.best_params_)



Best Regression Parameters: {'n_estimators': 200, 'min_samples_split': 10, 'max_depth': 10}
